# Estrutura do projeto

1. Dados

* Pode ser uma lista de listas ou um DataFrame
> Exemplo: altura, peso, idade

2. Transformação

* Calcular média por coluna
* Calcular desvio padrão
* Aplicar z-score

3. Análise

* Ver quais linhas têm valores muito altos/baixos
* Contar quantas anomalias por pessoa

In [2]:
import pandas as pd
import numpy as np

data = {
    "age": [22, 25, 30, 35, 40, 28, 32, 45, 50, 29],
    "salary": [2000, 2500, 3000, 4000, 5000, 2700, 3200, 7000, 12000, 2800],
    "years": [1, 2, 5, 10, 12, 3, 6, 15, 20, 2]
}


df = pd.DataFrame(data)

print(df)

   age  salary  years
0   22    2000      1
1   25    2500      2
2   30    3000      5
3   35    4000     10
4   40    5000     12
5   28    2700      3
6   32    3200      6
7   45    7000     15
8   50   12000     20
9   29    2800      2


In [3]:
### Média

def mean(xs):
    return sum(xs)/len(xs) if len(xs) > 0 else 0

Comparing with NumPy

In [4]:
np.mean(data["salary"])

np.float64(4420.0)

In [5]:
### Desvio Padrão
def std(xs):
    m = mean(xs)
    return (sum((x - m) ** 2 for x in xs) / len(xs) ** 0.5)

In [6]:
for name, content in data.items():
    print(f"{name} mean:", mean(content))

age mean: 33.6
salary mean: 4420.0
years mean: 7.6


In [7]:
def z_score(xs):
    m = mean(xs)
    s = std(xs)

    return [(x - m) /s for x in xs]

In [8]:
z_scores = {}

for name, content in df.items():
    z_scores[name] = z_score(content)
print(z_scores)

{'age': [-0.05106127624993487, -0.03785577377150343, -0.015846602974117722, 0.006162567823267994, 0.028171738620653707, -0.024650271293072006, -0.007042934655163435, 0.050180909418039425, 0.07219008021542514, -0.020248437133594865], 'salary': [-9.191784300960266e-05, -7.292655313158558e-05, -5.39352632535685e-05, -1.5952683497534343e-05, 2.2029896258499808e-05, -6.533003718037874e-05, -4.633874730236167e-05, 9.799505577056812e-05, 0.0002879079545507389, -6.153177920477533e-05], 'years': [-0.05634728012179077, -0.047809813436670956, -0.022197413381311513, 0.020489920044287558, 0.037564853414527186, -0.03927234675155114, -0.013659946696191699, 0.06317725346988663, 0.1058645868954857, -0.047809813436670956]}


### Checking anomalyies

In [29]:
### Z-Scores
df_z = (df - df.mean()) / df.std()

In [30]:
anomaly = (df_z.abs() > 2)
anomaly_dict = anomaly.to_dict(orient="list")

In [32]:
for i, person in enumerate(zip(
    anomaly_dict["age"],
    anomaly_dict["salary"],
    anomaly_dict["years"]
    )):
    qtd = sum(person)
    print(f'Person {i}: {qtd} anomalyies.')

Person 0: 0 anomalyies.
Person 1: 0 anomalyies.
Person 2: 0 anomalyies.
Person 3: 0 anomalyies.
Person 4: 0 anomalyies.
Person 5: 0 anomalyies.
Person 6: 0 anomalyies.
Person 7: 0 anomalyies.
Person 8: 1 anomalyies.
Person 9: 0 anomalyies.


### Pandas

In [33]:
for row in anomaly.values:
    sum(row)

In [34]:
anomaly.sum(axis=1)

0    0
1    0
2    0
3    0
4    0
5    0
6    0
7    0
8    1
9    0
dtype: int64

#### Or simply...

In [37]:
df["qtd_anomaly"] = anomaly.sum(axis=1)

In [38]:
df.sort_values("qtd_anomaly", ascending=False)

,age,salary,years,qtd_anomaly
8,50,12000,20,1
0,22,2000,1,0
1,25,2500,2,0
2,30,3000,5,0
4,40,5000,12,0
3,35,4000,10,0
5,28,2700,3,0
6,32,3200,6,0
7,45,7000,15,0
9,29,2800,2,0


#### Scores

In [39]:
df["score_anomaly"] = df_z.abs().sum(axis=1)

In [40]:
df["score_anomaly"]

0    3.122822
1    2.466767
2    1.275100
3    0.668897
4    1.592899
5    1.909348
6    0.829609
7    3.277745
8    6.260707
9    1.920420
Name: score_anomaly, dtype: float64